In [2]:
from IPython.display import display
import pandas as pd
import numpy as np
import scipy.sparse as sp

from sklearn.neighbors import NearestNeighbors

In [3]:
movies_df = pd.read_csv(
    "../../data/processed/movies_enriched.csv"
)

semantic_embeddings = np.load(
    "../../data/processed/semantic_embeddings.npy"
)

tfidf_matrix = sp.load_npz(
    "../../data/processed/tfidf_matrix.npz"
)

weighted_tfidf_matrix = sp.load_npz(
    "../../data/processed/weighted_tfidf_matrix.npz"
)

In [4]:
movies_df.shape

(70241, 20)

In [5]:
semantic_embeddings.shape

(70241, 384)

In [6]:
tfidf_matrix.shape

(70241, 5000)

In [7]:
weighted_tfidf_matrix.shape

(70241, 5000)

In [8]:
from sklearn.neighbors import NearestNeighbors

semantic_index = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

semantic_index.fit(semantic_embeddings)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [9]:
def get_semantic_candidates(movie_index, n=20):
    distances, indices = semantic_index.kneighbors(
        semantic_embeddings[movie_index].reshape(1, -1),
        n_neighbors=n + 1
    )

    return indices[0][1:], distances[0][1:]

In [10]:
movie_index = movies_df[
    movies_df["primaryTitle"].str.lower() == "interstellar"
].index[0]

semantic_indices, semantic_distances = get_semantic_candidates(
    movie_index,
    n=10
)

movies_df.loc[
    semantic_indices,
    ["primaryTitle", "startYear"]
].reset_index(drop=True)

,primaryTitle,startYear
0,The Prestige,2006
1,Tenet,2020
2,Return to the Lost World,1992
3,Time Runner,1993
4,Replicas,2018
5,Thru the Moebius Strip,2005
6,Supercollider,2013
7,Prometheus,2012
8,Chaos Walking,2021
9,Jurassic Galaxy,2018


In [11]:
from sklearn.neighbors import NearestNeighbors

tfidf_index = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

tfidf_index.fit(weighted_tfidf_matrix)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [12]:
def get_tfidf_candidates(movie_index, n=20):
    distances, indices = tfidf_index.kneighbors(
        weighted_tfidf_matrix[movie_index],
        n_neighbors=n + 1
    )

    return indices[0][1:], distances[0][1:]

In [13]:
tfidf_indices, tfidf_distances = get_tfidf_candidates(
    movie_index,
    n=10
)

movies_df.loc[
    tfidf_indices,
    ["primaryTitle", "startYear"]
].reset_index(drop=True)

,primaryTitle,startYear
0,The Mysterious Island of Captain Nemo,1973
1,MGS: Philanthropy,2009
2,Woken,2023
3,Mad Doctor of Blood Island,1968
4,Dead Sunrise,2017
5,Seat 25,2017
6,Spider-Man: Lotus,2023
7,Alien Visitor,1996
8,Black Garden,2019
9,Detroit: Become Human,2023


In [14]:
candidate_indices = list(
    set(semantic_indices) | set(tfidf_indices)
)

len(candidate_indices)

20

In [15]:
semantic_scores = {
    idx: 1 - distance
    for idx, distance in zip(
        semantic_indices,
        semantic_distances
    )
}

tfidf_scores = {
    idx: 1 - distance
    for idx, distance in zip(
        tfidf_indices,
        tfidf_distances
    )
}

In [16]:
hybrid_scores = {}

for idx in candidate_indices:

    semantic_score = semantic_scores.get(idx, 0)
    tfidf_score = tfidf_scores.get(idx, 0)

    hybrid_scores[idx] = (
        0.5 * semantic_score +
        0.5 * tfidf_score
    )

In [17]:
ranked_indices = sorted(
    hybrid_scores,
    key=hybrid_scores.get,
    reverse=True
)

In [18]:
movies_df.loc[
    ranked_indices,
    ["primaryTitle", "startYear"]
].reset_index(drop=True)

,primaryTitle,startYear
0,The Prestige,2006
1,Tenet,2020
2,Return to the Lost World,1992
3,Time Runner,1993
4,Replicas,2018
5,Thru the Moebius Strip,2005
6,Supercollider,2013
7,Prometheus,2012
8,Chaos Walking,2021
9,Jurassic Galaxy,2018


In [19]:
def hybrid_recommend(title, n=10):

    matches = movies_df[
        movies_df["primaryTitle"].str.lower() == title.lower()
    ]

    if matches.empty:
        return f"Movie '{title}' not found."

    movie_index = matches.index[0]

    semantic_indices, semantic_distances = get_semantic_candidates(
        movie_index,
        n=20
    )

    tfidf_indices, tfidf_distances = get_tfidf_candidates(
        movie_index,
        n=20
    )

    semantic_scores = {
        idx: 1 - distance
        for idx, distance in zip(
            semantic_indices,
            semantic_distances
        )
    }

    tfidf_scores = {
        idx: 1 - distance
        for idx, distance in zip(
            tfidf_indices,
            tfidf_distances
        )
    }
    candidate_indices = list(
        set(semantic_indices) | set(tfidf_indices)
    )

    hybrid_scores = {}

    for idx in candidate_indices:

        semantic_score = semantic_scores.get(idx, 0)
        tfidf_score = tfidf_scores.get(idx, 0)

        hybrid_scores[idx] = (
            0.5 * semantic_score +
            0.5 * tfidf_score
        )

    ranked_indices = sorted(
        hybrid_scores,
        key=hybrid_scores.get,
        reverse=True
    )

    return movies_df.loc[
        ranked_indices[:n],
        ["primaryTitle", "startYear"]
    ].reset_index(drop=True)

In [20]:
hybrid_recommend("Interstellar")

,primaryTitle,startYear
0,MGS: Philanthropy,2009
1,The Prestige,2006
2,Tenet,2020
3,Return to the Lost World,1992
4,Time Runner,1993
5,Replicas,2018
6,Thru the Moebius Strip,2005
7,Supercollider,2013
8,Prometheus,2012
9,Chaos Walking,2021


In [21]:
hybrid_recommend("Iron Man")

,primaryTitle,startYear
0,Iron Man 3,2013
1,Iron Man 2,2010
2,Spider-Man: Homecoming,2017
3,Avengers: Age of Ultron,2015
4,Captain America,1944
5,Universal Soldiers,2007
6,Doctor Strange,2016
7,Outlander,2008
8,Predator,1987
9,Fortress 2: Re-Entry,2000


In [22]:
hybrid_recommend("Avengers: Endgame")

,primaryTitle,startYear
0,Avengers: Infinity War,2018
1,Avengers: Age of Ultron,2015
2,X-Men: Apocalypse,2016
3,Captain America: Civil War,2016
4,Iron Man 3,2013
5,Iron Man,2008
6,Spider-Man: Homecoming,2017
7,X-Men: Days of Future Past,2014
8,Thor: The Dark World,2013
9,The Fantastic Four: First Steps,2025


In [23]:
hybrid_recommend("My Fault")

,primaryTitle,startYear
0,Mia,2023
1,My Fault: London,2025
2,Your Fault,2024
3,Culpa Nuestra,2025
4,Hazard,2022
5,Nothing Like the Holidays,2008
6,Igualita a mí,2010
7,Don't Blame Karma!,2022
8,All About My Mother,1999
9,Noah's Arc: Jumping the Broom,2008


In [24]:
hybrid_recommend("Fifty Shades of Grey")

,primaryTitle,startYear
0,Fifty Shades Darker,2017
1,Fifty Shades Freed,2018
2,At First Light,2018
3,Asylum,2005
4,Missionary,2013
5,Mom,2024
6,Crazy Kind of Love,2013
7,Return to Sender,2015
8,The Kate Logan Affair,2010
9,Eat,2014


In [25]:
hybrid_recommend("Thor: Love and Thunder")

,primaryTitle,startYear
0,Thor: Ragnarok,2017
1,Thor: The Dark World,2013
2,Ant-Man and the Wasp: Quantumania,2023
3,Thor,2011
4,Thor the Conqueror,1983
5,Journey 2: The Mysterious Island,2012
6,Guardians of the Galaxy Vol. 3,2023
7,Home,2015
8,Guardians of the Galaxy: Vol. 2,2017
9,The Expendables 2,2012


In [26]:
test_movies = [
    "Interstellar",
    "Inception",
    "Iron Man",
    "Avengers: Endgame",
    "Thor: Love and Thunder",
    "My Fault",
    "Fifty Shades of Grey"
]

In [27]:
for movie in test_movies:
    print("\n" + "=" * 60)
    print(f"🎬 {movie}")
    print("=" * 60)

    display(hybrid_recommend(movie))


🎬 Interstellar


,primaryTitle,startYear
0,MGS: Philanthropy,2009
1,The Prestige,2006
2,Tenet,2020
3,Return to the Lost World,1992
4,Time Runner,1993
5,Replicas,2018
6,Thru the Moebius Strip,2005
7,Supercollider,2013
8,Prometheus,2012
9,Chaos Walking,2021



🎬 Inception


,primaryTitle,startYear
0,Baida,2025
1,Tenet,2020
2,Sparks,2013
3,Purple Noon,1960
4,8MM,1999
5,"The Case Is Closed, Forget It",1971
6,Extracted,2012
7,Lying and Stealing,2019
8,Cypher,2002
9,The Cold Light of Day,2012



🎬 Iron Man


...

,primaryTitle,startYear
0,Thor: Ragnarok,2017
1,Thor: The Dark World,2013
2,Ant-Man and the Wasp: Quantumania,2023
3,Thor,2011
4,Thor the Conqueror,1983
5,Journey 2: The Mysterious Island,2012
6,Guardians of the Galaxy Vol. 3,2023
7,Home,2015
8,Guardians of the Galaxy: Vol. 2,2017
9,The Expendables 2,2012



🎬 My Fault


,primaryTitle,startYear
0,Mia,2023
1,My Fault: London,2025
2,Your Fault,2024
3,Culpa Nuestra,2025
4,Hazard,2022
5,Nothing Like the Holidays,2008
6,Igualita a mí,2010
7,Don't Blame Karma!,2022
8,All About My Mother,1999
9,Noah's Arc: Jumping the Broom,2008



🎬 Fifty Shades of Grey


,primaryTitle,startYear
0,Fifty Shades Darker,2017
1,Fifty Shades Freed,2018
2,At First Light,2018
3,Asylum,2005
4,Missionary,2013
5,Mom,2024
6,Crazy Kind of Love,2013
7,Return to Sender,2015
8,The Kate Logan Affair,2010
9,Eat,2014


In [28]:
semantic_indices, semantic_distances = get_semantic_candidates(
    movie_index,
    n=10
)

movies_df.loc[
    semantic_indices,
    ["primaryTitle", "startYear"]
].reset_index(drop=True)

,primaryTitle,startYear
0,The Prestige,2006
1,Tenet,2020
2,Return to the Lost World,1992
3,Time Runner,1993
4,Replicas,2018
5,Thru the Moebius Strip,2005
6,Supercollider,2013
7,Prometheus,2012
8,Chaos Walking,2021
9,Jurassic Galaxy,2018


In [29]:
tfidf_indices, tfidf_distances = get_tfidf_candidates(
    movie_index,
    n=10
)

movies_df.loc[
    tfidf_indices,
    ["primaryTitle", "startYear"]
].reset_index(drop=True)

,primaryTitle,startYear
0,The Mysterious Island of Captain Nemo,1973
1,MGS: Philanthropy,2009
2,Woken,2023
3,Mad Doctor of Blood Island,1968
4,Dead Sunrise,2017
5,Seat 25,2017
6,Spider-Man: Lotus,2023
7,Alien Visitor,1996
8,Black Garden,2019
9,Detroit: Become Human,2023


In [30]:
candidate_indices = list(
    set(semantic_indices) | set(tfidf_indices)
)

diagnostic_df = movies_df.loc[
    candidate_indices,
    ["primaryTitle", "startYear"]
].copy()

diagnostic_df["semantic_distance"] = np.nan
diagnostic_df["tfidf_distance"] = np.nan

for i, idx in enumerate(semantic_indices):
    diagnostic_df.loc[idx, "semantic_distance"] = semantic_distances[i]

for i, idx in enumerate(tfidf_indices):
    diagnostic_df.loc[idx, "tfidf_distance"] = tfidf_distances[i]

diagnostic_df.sort_values(
    by=["semantic_distance", "tfidf_distance"],
    na_position="last"
).reset_index(drop=True)

,primaryTitle,startYear,semantic_distance,tfidf_distance
0,The Prestige,2006,0.324438,NaN
1,Tenet,2020,0.329538,NaN
2,Return to the Lost World,1992,0.357730,NaN
3,Time Runner,1993,0.359369,NaN
4,Replicas,2018,0.363981,NaN
5,Thru the Moebius Strip,2005,0.368845,NaN
6,Supercollider,2013,0.369215,NaN
7,Prometheus,2012,0.372560,NaN
8,Chaos Walking,2021,0.374401,NaN
9,Jurassic Galaxy,2018,0.374647,NaN


In [31]:
diagnostic_df.sort_values(
    by="tfidf_distance",
    na_position="last"
).reset_index(drop=True)

,primaryTitle,startYear,semantic_distance,tfidf_distance
0,The Mysterious Island of Captain Nemo,1973,NaN,0.481630
1,MGS: Philanthropy,2009,NaN,0.521485
2,Mad Doctor of Blood Island,1968,NaN,0.528074
3,Woken,2023,NaN,0.528074
4,Dead Sunrise,2017,NaN,0.528074
5,Alien Visitor,1996,NaN,0.531604
6,Seat 25,2017,NaN,0.531604
7,Spider-Man: Lotus,2023,NaN,0.531604
8,Detroit: Become Human,2023,NaN,0.548538
9,Black Garden,2019,NaN,0.548538


In [32]:
diagnostic_df.sort_values(
    by="semantic_distance",
    na_position="last"
).reset_index(drop=True)

,primaryTitle,startYear,semantic_distance,tfidf_distance
0,The Prestige,2006,0.324438,NaN
1,Tenet,2020,0.329538,NaN
2,Return to the Lost World,1992,0.357730,NaN
3,Time Runner,1993,0.359369,NaN
4,Replicas,2018,0.363981,NaN
5,Thru the Moebius Strip,2005,0.368845,NaN
6,Supercollider,2013,0.369215,NaN
7,Prometheus,2012,0.372560,NaN
8,Chaos Walking,2021,0.374401,NaN
9,Jurassic Galaxy,2018,0.374647,NaN


In [33]:
import inspect

print(inspect.getsource(hybrid_recommend))

def hybrid_recommend(title, n=10):

    matches = movies_df[
        movies_df["primaryTitle"].str.lower() == title.lower()
    ]

    if matches.empty:
        return f"Movie '{title}' not found."

    movie_index = matches.index[0]

    semantic_indices, semantic_distances = get_semantic_candidates(
        movie_index,
        n=20
    )

    tfidf_indices, tfidf_distances = get_tfidf_candidates(
        movie_index,
        n=20
    )

    semantic_scores = {
        idx: 1 - distance
        for idx, distance in zip(
            semantic_indices,
            semantic_distances
        )
    }

    tfidf_scores = {
        idx: 1 - distance
        for idx, distance in zip(
            tfidf_indices,
            tfidf_distances
        )
    }
    candidate_indices = list(
        set(semantic_indices) | set(tfidf_indices)
    )

    hybrid_scores = {}

    for idx in candidate_indices:

        semantic_score = semantic_scores.get(idx, 0)
        tfidf_score = tfidf_scores.get(idx

In [34]:
source = inspect.getsource(hybrid_recommend)

lines = source.splitlines()

for i, line in enumerate(lines[:45], 1):
    print(f"{i}: {line}")

1: def hybrid_recommend(title, n=10):
2: 
3:     matches = movies_df[
4:         movies_df["primaryTitle"].str.lower() == title.lower()
5:     ]
6: 
7:     if matches.empty:
8:         return f"Movie '{title}' not found."
9: 
10:     movie_index = matches.index[0]
11: 
12:     semantic_indices, semantic_distances = get_semantic_candidates(
13:         movie_index,
14:         n=20
15:     )
16: 
17:     tfidf_indices, tfidf_distances = get_tfidf_candidates(
18:         movie_index,
19:         n=20
20:     )
21: 
22:     semantic_scores = {
23:         idx: 1 - distance
24:         for idx, distance in zip(
25:             semantic_indices,
26:             semantic_distances
27:         )
28:     }
29: 
30:     tfidf_scores = {
31:         idx: 1 - distance
32:         for idx, distance in zip(
33:             tfidf_indices,
34:             tfidf_distances
35:         )
36:     }
37:     candidate_indices = list(
38:         set(semantic_indices) | set(tfidf_indices)
39:     )
40: 
41:

In [35]:
for i, line in enumerate(lines[25:45], 26):
    print(f"{i}: {line}")

26:             semantic_distances
27:         )
28:     }
29: 
30:     tfidf_scores = {
31:         idx: 1 - distance
32:         for idx, distance in zip(
33:             tfidf_indices,
34:             tfidf_distances
35:         )
36:     }
37:     candidate_indices = list(
38:         set(semantic_indices) | set(tfidf_indices)
39:     )
40: 
41:     hybrid_scores = {}
42: 
43:     for idx in candidate_indices:
44: 
45:         semantic_score = semantic_scores.get(idx, 0)


In [36]:
for i, line in enumerate(lines[45:60], 46):
    print(f"{i}: {line}")

46:         tfidf_score = tfidf_scores.get(idx, 0)
47: 
48:         hybrid_scores[idx] = (
49:             0.5 * semantic_score +
50:             0.5 * tfidf_score
51:         )
52: 
53:     ranked_indices = sorted(
54:         hybrid_scores,
55:         key=hybrid_scores.get,
56:         reverse=True
57:     )
58: 
59:     return movies_df.loc[
60:         ranked_indices[:n],


In [37]:
movie = "Interstellar"

movie_index = movies_df[
    movies_df["primaryTitle"].str.lower() == movie.lower()
].index[0]

semantic_indices, semantic_distances = get_semantic_candidates(
    movie_index,
    n=20
)

tfidf_indices, tfidf_distances = get_tfidf_candidates(
    movie_index,
    n=20
)

semantic_scores = {
    idx: 1 - distance
    for idx, distance in zip(
        semantic_indices,
        semantic_distances
    )
}

tfidf_scores = {
    idx: 1 - distance
    for idx, distance in zip(
        tfidf_indices,
        tfidf_distances
    )
}

candidate_indices = list(
    set(semantic_indices) | set(tfidf_indices)
)

debug_df = movies_df.loc[
    candidate_indices,
    ["primaryTitle", "startYear"]
].copy()

debug_df["semantic_score"] = [
    semantic_scores.get(idx, 0)
    for idx in candidate_indices
]

debug_df["tfidf_score"] = [
    tfidf_scores.get(idx, 0)
    for idx in candidate_indices
]

debug_df["hybrid_score"] = (
    0.5 * debug_df["semantic_score"]
    + 0.5 * debug_df["tfidf_score"]
)

debug_df.sort_values(
    "hybrid_score",
    ascending=False
).reset_index(drop=True)

,primaryTitle,startYear,semantic_score,tfidf_score,hybrid_score
0,MGS: Philanthropy,2009,0.616064,0.478515,0.547290
1,The Prestige,2006,0.675562,0.000000,0.337781
2,Tenet,2020,0.670462,0.000000,0.335231
3,Return to the Lost World,1992,0.642270,0.000000,0.321135
4,Time Runner,1993,0.640631,0.000000,0.320316
5,Replicas,2018,0.636019,0.000000,0.318009
6,Thru the Moebius Strip,2005,0.631155,0.000000,0.315578
7,Supercollider,2013,0.630785,0.000000,0.315392
8,Prometheus,2012,0.627440,0.000000,0.313720
9,Chaos Walking,2021,0.625599,0.000000,0.312800


In [38]:
print("SEMANTIC TOP 20")
display(
    movies_df.loc[
        semantic_indices,
        ["primaryTitle", "startYear"]
    ].reset_index(drop=True)
)

print("\nTF-IDF TOP 20")
display(
    movies_df.loc[
        tfidf_indices,
        ["primaryTitle", "startYear"]
    ].reset_index(drop=True)
)

SEMANTIC TOP 20


,primaryTitle,startYear
0,The Prestige,2006
1,Tenet,2020
2,Return to the Lost World,1992
3,Time Runner,1993
4,Replicas,2018
5,Thru the Moebius Strip,2005
6,Supercollider,2013
7,Prometheus,2012
8,Chaos Walking,2021
9,Jurassic Galaxy,2018



TF-IDF TOP 20


,primaryTitle,startYear
0,The Mysterious Island of Captain Nemo,1973
1,MGS: Philanthropy,2009
2,Woken,2023
3,Mad Doctor of Blood Island,1968
4,Dead Sunrise,2017
5,Seat 25,2017
6,Spider-Man: Lotus,2023
7,Alien Visitor,1996
8,Black Garden,2019
9,Detroit: Become Human,2023


In [39]:
movie_index = movies_df[
    movies_df["primaryTitle"].str.lower() == "interstellar"
].index[0]

print("INTERSTELLAR INDEX:", movie_index)

print("\nTF-IDF TEXT:")
print(movies_df.loc[movie_index, "content"])

INTERSTELLAR INDEX: 31735

TF-IDF TEXT:


KeyError: 'content'

In [ ]:
print(movies_df.columns.tolist())

['tconst', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'directors', 'writers', 'castName', 'poster_path', 'backdrop_path', 'overview', 'tmdb_id', 'popularity', 'vote_average', 'vote_count', 'release_date']


In [ ]:
print(movies_df.loc[movie_index, "overview"])

The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.


In [ ]:
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1610451 stored elements and shape (70241, 5000)>
  Coords	Values
  (0, 1310)	0.10219148991434929
  (0, 3060)	0.1661067632730908
  (0, 4195)	0.13360169966904836
  (0, 2607)	0.1578982992651739
  (0, 1674)	0.23532762652783548
  (0, 1679)	0.42651998398861235
  (0, 2764)	0.20555886108734983
  (0, 15)	0.20280885631521
  (0, 4723)	0.24708154718912362
  (0, 3453)	0.1597558072842395
  (0, 353)	0.20891228110656687
  (0, 4486)	0.11378499673042956
  (0, 676)	0.17524786234468895
  (0, 3617)	0.1615901488331118
  (0, 1490)	0.1559437529215103
  (0, 4076)	0.22249082810659065
  (0, 1042)	0.14445623161523619
  (0, 4954)	0.095181703509634
  (0, 1693)	0.11553801974864202
  (0, 2530)	0.14437185516586565
  (0, 1648)	0.1560296305765066
  (0, 3710)	0.1543208763591418
  (0, 1728)	0.16407088214810822
  (0, 2574)	0.13127864817725354
  (0, 74)	0.20074789770552678
  :	:
  (70239, 563)	0.21253907859116217
  (70239, 1305)	0.20885605824366868
  (70239, 796)

In [ ]:
print("MOVIE INDEX:", movie_index)
print("MOVIE:", movies_df.loc[movie_index, "primaryTitle"])
print("TF-IDF SHAPE:", tfidf_matrix[movie_index].shape)
print("NON-ZERO FEATURES:", tfidf_matrix[movie_index].nnz)

MOVIE INDEX: 31735
MOVIE: Interstellar
TF-IDF SHAPE: (1, 5000)
NON-ZERO FEATURES: 20


In [ ]:
print("MOVIES:", len(movies_df))
print("TF-IDF ROWS:", tfidf_matrix.shape[0])
print("SEMANTIC ROWS:", semantic_embeddings.shape[0])

MOVIES: 70241
TF-IDF ROWS: 70241
SEMANTIC ROWS: 70241


In [ ]:
print("SEMANTIC CANDIDATES:", len(semantic_indices))
print("TF-IDF CANDIDATES:", len(tfidf_indices))

print(
    "OVERLAP:",
    len(set(semantic_indices) & set(tfidf_indices))
)

print(
    "SEMANTIC ONLY:",
    len(set(semantic_indices) - set(tfidf_indices))
)

print(
    "TF-IDF ONLY:",
    len(set(tfidf_indices) - set(semantic_indices))
)

SEMANTIC CANDIDATES: 20
TF-IDF CANDIDATES: 20
OVERLAP: 1
SEMANTIC ONLY: 19
TF-IDF ONLY: 19


In [ ]:
print("SEMANTIC ONLY:")
print(
    movies_df.loc[
        list(set(semantic_indices) - set(tfidf_indices)),
        ["primaryTitle", "startYear"]
    ].to_string(index=False)
)

print("\nTF-IDF ONLY:")
print(
    movies_df.loc[
        list(set(tfidf_indices) - set(semantic_indices)),
        ["primaryTitle", "startYear"]
    ].to_string(index=False)
)

print("\nOVERLAP:")
print(
    movies_df.loc[
        list(set(semantic_indices) & set(tfidf_indices)),
        ["primaryTitle", "startYear"]
    ].to_string(index=False)
)

SEMANTIC ONLY:
              primaryTitle  startYear
             Supercollider       2013
         A Minecraft Movie       2025
           Mission to Mars       2000
1492: Conquest of Paradise       1992
              The Sea Wolf       1941
               Time Runner       1993
          The Osiris Child       2016
                  Replicas       2018
                      Xeno       2025
                Prometheus       2012
                  Ad Astra       2019
                     Tenet       2020
  Return to the Lost World       1992
              The Prestige       2006
             Chaos Walking       2021
           Jurassic Galaxy       2018
                  Top Line       1988
    Thru the Moebius Strip       2005
                  Spectral       2016

TF-IDF ONLY:
                                              primaryTitle  startYear
                                               Planeta bur       1962
                                                 Timequest       2000
 

In [ ]:
tfidf_only = list(
    set(tfidf_indices) - set(semantic_indices)
)

print("TF-IDF ONLY COUNT:", len(tfidf_only))

for idx in tfidf_only:
    print(
        idx,
        "->",
        movies_df.loc[idx, "primaryTitle"],
        "| score:",
        tfidf_scores.get(idx, 0)
    )

TF-IDF ONLY COUNT: 19
6656 -> Planeta bur | score: 0.43124889308908865
23682 -> Timequest | score: 0.4514624924794284
47747 -> Woken | score: 0.4719261847531683
6535 -> The Day Mars Invaded Earth | score: 0.4514624924794284
39956 -> Spider-Man: Lotus | score: 0.4683960089838741
33559 -> InAlienable | score: 0.4514624924794284
18843 -> Alien Visitor | score: 0.4683960089838741
43554 -> Hooked 2: Next Level | score: 0.43128294614895246
38188 -> Helsreach: The Movie | score: 0.4514624924794284
63795 -> Seat 25 | score: 0.4683960089838741
9660 -> The Mysterious Island of Captain Nemo | score: 0.5183699711308077
49474 -> Miracle Mirrors | score: 0.4420791106726113
13644 -> The Adventures of Buckaroo Banzai Across the 8th Dimension | score: 0.4500519533956483
4323 -> Cat-Women of the Moon | score: 0.44725023932995867
52964 -> Detroit: Become Human | score: 0.4514624924794284
7527 -> Agent for H.A.R.M. | score: 0.433071470735533
8299 -> Mad Doctor of Blood Island | score: 0.4719261847531683
6